In [ ]:
import pandas as pd
import time
import requests
import csv
import json
import ast
import re
import numpy as np
import html
import networkx as nx

import polars as pl

from datasketch import MinHash, MinHashLSH, LeanMinHash
import kshingle as ks
import pickle
from tqdm import tqdm
import ahocorasick

import unicodedata

def remove_accents(text):
    # Normalize the string to decompose characters with accents
    normalized_text = unicodedata.normalize('NFD', text)
    # Filter out accent marks
    return ''.join(char for char in normalized_text if unicodedata.category(char) != 'Mn')


def get_shingles(text, shingle_length, wildcard=False):
    shingles = ks.shingleset_k(text, shingle_length)
    if wildcard:
        shingles = shingles.union(ks.wildcard_shinglesets(shingles, 2))
    return shingles


# Define a function to create a MinHash object from a string
def create_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for word in get_shingles(text, 3, wildcard=False):
        m.update(word.encode('utf-16', 'surrogatepass').decode('utf-16').encode('utf-8'))
    return LeanMinHash(m)

mailto = ""

def query_openalex_funders(search_string, abbrev = False):
    if abbrev:
        search_string = f'"{search_string}"'
    url = f'https://api.openalex.org/funders?mailto={mailto}&search={search_string}'
    response = requests.get(url)
    time.sleep(0.1)
    if response.status_code == 200:
        data = response.json()
        if 'results' in data and len(data['results']) > 0:
            result = data['results']
            for r in result:
                r['search_string'] = search_string
            return result
        else:
            return [{"search_string": search_string,
                "error": "No results found"}]
    else:
        print(f"Error fetching data for batch. Status code: {response.status_code}")
        return [{"search_string": search_string,
            "error": "Failed to fetch data"}]

def remove_surrogates(s):
    if isinstance(s, str):
        return s.encode('utf-8', 'replace').decode('utf-8')
    return s


tqdm.pandas()

In [7]:

with open("openalex_institutions/funders_core+basic+ids+extra+counts.json") as f:
    funder_schema = json.load(f)

dtype_dict = {k: v['type'] for k, v in funder_schema['schema'].items()}
na_values = {k: v['NA value'] for k, v in funder_schema['schema'].items()}
    
filepath = "openalex_institutions/funders_core+basic+ids+extra+counts.tsv"

funders = pd.read_csv(filepath, sep="\t", encoding='utf-8', dtype=dtype_dict, na_values=na_values, doublequote=False,
             escapechar="\\",on_bad_lines="warn", low_memory=False, quoting=csv.QUOTE_MINIMAL)

funders['alternate_titles'] = funders['alternate_titles'].apply(ast.literal_eval)

with open("openalex_institutions/institutions_core+basic+alternate_names+ids+geo+urls+roles.json") as f:
    institution_schema = json.load(f)

dtype_dict = {k: v['type'] for k, v in institution_schema['schema'].items()}
na_values = {k: v['NA value'] for k, v in institution_schema['schema'].items()}

filepath = "openalex_institutions/institutions_core+basic+alternate_names+ids+geo+urls+roles.tsv"

institutions = pd.read_csv(filepath, sep="\t", encoding='utf-8', dtype=dtype_dict, na_values=na_values, doublequote=False,
             escapechar="\\",on_bad_lines="warn", low_memory=False, quoting=csv.QUOTE_MINIMAL)

institutions['display_name_alternatives'] = institutions['display_name_alternatives'].apply(ast.literal_eval)
institutions['display_name_acronyms'] = institutions['display_name_acronyms'].apply(ast.literal_eval)


In [8]:
#u_agencies = pd.read_feather('wos_agenciesWithCounts.feather')

u_agencies = pl.read_ipc('WOS_TSV_FUNDING_2023.feather')
u_agencies = (u_agencies
.with_columns(agency_name = pl.col("agency_name").str.split(";"))
.explode("agency_name")
.with_columns(agency_name = pl.col("agency_name").str.split(":").list.first().str.strip_chars())
.filter(pl.col('agency_name') != "")
)['agency_name'].value_counts(sort=True).rename({'agency_name':'grant_agency', 'count':'counts'}).to_pandas()

u_agencies['grant_agency'] = u_agencies['grant_agency'].apply(html.unescape)
u_agencies['grant_agency'] = u_agencies['grant_agency'].str.replace('&', 'and')
u_agencies['grant_agency'] = u_agencies['grant_agency'].str.replace("''", "'")

u_agencies['grant_agency_lower'] = u_agencies['grant_agency'].str.lower()
u_agencies['grant_agency_lower'] = u_agencies['grant_agency_lower'].str.strip()

u_agencies[['abbrev']] = u_agencies.grant_agency.str.extract(r'\(([A-Z]*)\)')
u_agencies.shape

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


(7926557, 4)

### Preparing data for document clustering (no need to run)

In [4]:
minhash_file = 'grant_agency_minhashes.pkl'

with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(u_agencies.index, u_agencies.grant_agency), total=len(u_agencies), desc="Querying grant agency"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)

Querying grant agency: 100%|██████████| 7926557/7926557 [2:02:10<00:00, 1081.30it/s]  


In [5]:
minhash_file = 'grant_agency_minhashes.pkl'
# Create the LSH object with threshold for near-duplicates
lsh = MinHashLSH(threshold=0.95, num_perm=128)

with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            lsh.insert(f"{idx}", lean_minhash)
    except EOFError:
        pass 



In [6]:
edge_list_file = 'edge_list.csv'

with open(minhash_file, 'rb') as f_in, open(edge_list_file, 'w', newline='') as f_out:
    writer = csv.writer(f_out)
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            result = lsh.query(lean_minhash)
            if result:
                for res in result:
                    writer.writerow([idx, res])
    except EOFError:
        pass

### Read outputs from document clustering

edge_list is an edge list of the graph induced by calculating document similarity between all documents (i.e. all funder names). Two documents are linked if jaccard similarity is above 0.95.

In [9]:
edge_list_file = 'edge_list.csv'

#read edgelist from file
G = nx.read_edgelist(edge_list_file, delimiter=',', nodetype=int)

G.remove_edges_from(nx.selfloop_edges(G))
#connected components
components = list(nx.connected_components(G))
len(components)

7297223

In [10]:
#map components to ids in the original dataframe
component_map = {}

for idx, component in enumerate(components):
    for node in component:
        component_map[node] = idx

u_agencies['component'] = u_agencies.index.map(component_map)

### Preprocess openalex data

This block preproccesses both OA funder and institution data and combines them
* combine names (canonical names, alternate names)
* replace & with and
* fix encoding issues
    * some characters (emojis) need to be encoded in utf-16 otherwise the deduplication algorithm doesn't work
* remove accents - WOS data is ASCII which has no accents
* openalex has inconsistent use of apostrophes
* if funders and institutions share canonical name they also share acronyms
* if duplicates are present we keep the funder data and supplement with institution data


In [11]:
funders['all_names'] = funders.apply(lambda x: [x['display_name']] + x['alternate_titles'], axis=1)
funders['all_names'] = funders['all_names'].apply(lambda x: [y.replace('&', 'and') for y in x])
funders['all_names'] = funders['all_names'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])

funders['all_names'] = funders['all_names'].apply(
    lambda x: [remove_accents(y) for y in x])

funders['all_names'] = funders['all_names'].apply(
    lambda x: [y.replace("’", "'") for y in x])


funders['all_names_lower'] = funders['all_names'].apply(lambda x: [y.lower() for y in x])
funders['all_names_lower'] = funders['all_names_lower'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])
funders['all_names_lower'] = funders['all_names_lower'].apply(
    lambda x: [y.strip() for y in x])

funders['display_name'] = funders['display_name'].apply(remove_accents)
funders['display_name_lower'] = funders['display_name'].str.lower()
funders['display_name_lower'] = funders['display_name_lower'].apply(
    lambda x: x.encode('utf-16', 'surrogatepass').decode('utf-16'))
funders['display_name_lower'] = funders['display_name_lower'].str.strip()

funders.sort_values('grants_count', ascending=False, inplace=True)

institutions['all_names'] = institutions.apply(lambda x: [x['display_name']] + x['display_name_acronyms'] + x['display_name_alternatives'], axis=1)
institutions['all_names'] = institutions['all_names'].apply(lambda x: [y.replace('&', 'and') for y in x])
institutions['all_names'] = institutions['all_names'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])
institutions['all_names'] = institutions['all_names'].apply(
    lambda x: [remove_accents(y) for y in x])
institutions['all_names'] = institutions['all_names'].apply(
    lambda x: [y.replace("’", "'") for y in x])

institutions['all_names_lower'] = institutions['all_names'].apply(lambda x: [y.lower() for y in x])
institutions['all_names_lower'] = institutions['all_names_lower'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])
institutions['all_names_lower'] = institutions['all_names_lower'].apply(
    lambda x: [y.strip() for y in x])


institutions['display_name'] = institutions['display_name'].apply(remove_accents)
institutions['display_name_lower'] = institutions['display_name'].str.lower()
institutions['display_name_lower'] = institutions['display_name_lower'].apply(
    lambda x: x.encode('utf-16', 'surrogatepass').decode('utf-16'))
institutions['display_name_lower'] = institutions['display_name_lower'].str.strip()

institutions['display_name_acronyms'] = institutions['display_name_acronyms'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])
institutions['display_name_acronyms'] = institutions['display_name_acronyms'].apply(
    lambda x: [y.strip() for y in x])

abbrev_map = institutions.set_index('display_name')['display_name_acronyms'].to_dict()
funders['display_name_acronyms'] = funders['display_name'].map(abbrev_map)

ror_map = institutions.set_index('display_name')['ids:ror'].to_dict()
funders['ids:ror'] = funders['ids:ror'].fillna(funders.display_name.map(ror_map))

funders['source'] = 'funders'
institutions['source'] = 'institutions'

institutions = pd.concat([funders, institutions], ignore_index=True)
print(institutions.shape)

institutions[['all_names', 'all_names_lower','display_name_acronyms', ]] = (institutions.groupby('display_name')
 [['all_names', 'all_names_lower','display_name_acronyms', ]]
 .transform(sum))
institutions['display_name_acronyms'] = institutions['display_name_acronyms'].apply(lambda x: x if x else [])

institutions['all_names'] = institutions['all_names'].apply(lambda x: list(set(x)))
institutions['all_names_lower'] = institutions['all_names_lower'].apply(lambda x: list(set(x)))
institutions['display_name_acronyms'] = institutions['display_name_acronyms'].apply(lambda x: list(set(x)))

institutions.drop_duplicates(subset=['display_name'], inplace=True)
print(institutions.shape)

(141696, 33)


/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_2275/2496192323.py:68: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .transform(sum))


(129397, 33)


In [12]:
ror = pd.read_csv('v1.55-2024-10-31-ror-data_schema_v2.csv')
in_dimensions = pl.read_parquet('funders_dimensions.parquet').with_columns(pl.lit('True').alias('in_dimensions')).to_pandas()

ror = ror.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')

institutions_ = institutions.merge(ror[['id','external_ids.type.grid.preferred']], left_on="ids:ror", right_on='id', how='left', suffixes=("","_ror"))
institutions_ = institutions_.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')
institutions_ = institutions_.merge(in_dimensions, how='left', left_on="ids:grid", right_on='funder', suffixes=("_ror","_oa"))

print(institutions_.assign(funder = institutions_.funder_ror.fillna(institutions_.funder_oa)).funder_ror.nunique())

institutions_ = institutions_.rename(columns={'funder_ror':'dimensions_grid', })

/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_2275/651156140.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  ror = pd.read_csv('v1.55-2024-10-31-ror-data_schema_v2.csv')


637


In [13]:
institutions_[['id', 'display_name', 'country_code', 'ids:ror', 'homepage_url',
'source', 'geo:city', 'geo:geonames_city_id', 'geo:region', 'geo:country_code',
       'geo:country', 'geo:latitude', 'geo:longitude', 'dimensions_grid']].to_parquet('oa_institutions.parquet')

### Exact match

In [14]:
u_agencies = u_agencies.merge(institutions[['display_name_lower','display_name', 'id']],
    left_on='grant_agency_lower', right_on='display_name_lower', how='left')

# if id not na then "display_exact" use fillna approach
u_agencies.loc[~u_agencies['id'].isna(), 'match_type'] = 'display_exact'

u_agencies = u_agencies.merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
    left_on='grant_agency_lower', right_on='all_names_lower', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'all_exact'


In [15]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(7420729, 10)
0.9857306472180833


0.5521457474578482

### Apply manual annotations by Jacob

In [16]:
#load hhs map from json
with open('hhs_map.json') as f:
    HHS_map = json.load(f)
HHS_map = {k.strip():v for k, v in HHS_map.items()}

with open('min_100_map.json') as f:
    min_100_map = json.load(f)
min_100_map = {k.strip():v for k, v in min_100_map.items()}

p_1000 = pd.read_csv("plus_1000_labelled.csv", sep=";")
p_1000 = p_1000.assign(id = p_1000.id.str.slice(1),
                       grant_agency = p_1000.grant_agency.apply(
                        lambda x: x[1:-1] if x.startswith('"') and x.endswith('"') else x)
).dropna().set_index('grant_agency')['id'].to_dict()


In [17]:
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(HHS_map))
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(min_100_map))
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(p_1000))

### More manual annotations

In [18]:
last_dict = {'Korean Government (MOEHRD)':'F4320327953',
'Scientific and Technical Research Council of Turkey (TUBITAK)':'F4320322626',
'Strategic Priority Research Program (B) of the Chinese Academy of Sciences' : 'F4320321133',
'National High Technology Research and Development Program ("863" Program) of China':'F4320335773',
'Soft and Hybrid Nanotechnology Experimental (SHyNE) Resource':'F4320332170'}

last_dict['MPG, Germany'] = 'https://openalex.org/F4320322434'
last_dict['GlaxoSmithKline Biologicals SA'] = 'https://openalex.org/F4320307773'
last_dict['Merck Research Laboratories'] = 'https://openalex.org/F4320307776'
last_dict['Spanish Participation Group'] = 'https://openalex.org/F4320322930'
last_dict['CNCSIS'] = 'https://openalex.org/F4320322505'
last_dict['Siemens Healthcare'] = 'https://openalex.org/F4320327217'
last_dict['Ruth L. Kirschstein National Research Service Award'] = 'https://openalex.org/F4320332161'
last_dict['MIUR-PRIN'] = 'https://openalex.org/F4320321873'
last_dict['State Key Research Development Program of China'] = 'https://openalex.org/F4320335777'
last_dict['COLCIENCIAS, Colombia'] = 'https://openalex.org/F4320309955'
last_dict['Korean Scientist Group'] = 'https://openalex.org/F4320322349'
last_dict['ARO MURI'] = 'https://openalex.org/F4320333591'
last_dict['University Grant Commission'] = 'https://openalex.org/F4320320767'
last_dict['Centocor'] = 'https://openalex.org/F4320337601'
last_dict['German Participation Group'] = 'https://openalex.org/F4320331606'
last_dict['Interuniversity Attraction Poles Programme'] = 'https://openalex.org/F4320321454'
last_dict['863 Project'] = 'https://openalex.org/F4320335773'
last_dict['NUPTSF'] = 'https://openalex.org/F4320321001'
last_dict['Austrian Science Foundation'] = 'https://openalex.org/F4320321181'
last_dict['Anhui Initiative in Quantum Information Technologies'] = 'https://openalex.org/F4320334897'
last_dict['Vanier Canada Graduate Scholarship'] = 'https://openalex.org/F4320319880'
last_dict['Brazilian Participation Group'] = 'https://openalex.org/F4320321992'
last_dict['French Participation Group'] = 'https://openalex.org/F4320320883'
last_dict['Japan Participation Group'] = 'https://openalex.org/F4320323772'
#last_dict['Neurotrack Technologies'] = 
last_dict['National Natural Science Fundation of China'] = 'https://openalex.org/F4320321001'
last_dict['Nebraska Research Initiative'] = 'https://openalex.org/F4320310264'
last_dict['DIVISION OF CANCER TREATMENT'] = 'https://openalex.org/F4320309684'
#last_dict['EuroImmun'] = 'https://openalex.org/F4320335322'
last_dict['National Key RD Plan'] = 'https://openalex.org/F4320335777'
last_dict['University Grant Commission, New Delhi'] = 'https://openalex.org/F4320320767'
last_dict['NIH-NIGMS'] = 'https://openalex.org/F4320337354'
last_dict['Sloan Digital Sky Survey (SDSS)'] = 'https://openalex.org/F4320306151'
last_dict['Commission of the European Communities'] = 'https://openalex.org/F4320320300'
last_dict['Icelandic Research Fund'] = 'https://openalex.org/F4320321020'
last_dict['Hong Kong Scholars Program'] = 'https://openalex.org/F4320323540'
last_dict['National Magnetic Confinement Fusion Science Program of China'] = 'https://openalex.org/F4320327893'
last_dict['Taishan Scholars Program of Shandong Province'] = 'https://openalex.org/F4320326189'
last_dict['Fundamental Research Grant Scheme'] = 'https://openalex.org/F4320321709'
last_dict['MICINN, Spain'] = 'https://openalex.org/F4320322930'
last_dict['NASA Office of Space Science'] = 'https://openalex.org/F4320306101'
last_dict['Parkinson"s UK'] = 'https://openalex.org/F4320320021'
last_dict['Fundamental Research Grant Scheme (FRGS)'] = 'https://openalex.org/F4320321709'
last_dict['Elan Pharmaceuticals, Inc.'] = 'https://openalex.org/F4320321667'
last_dict['CONICYT, Chile'] = 'https://openalex.org/F4320334812'
last_dict['Australian Ctr Intl Agr Res'] = 'https://openalex.org/F4320334709'
last_dict['JST-CREST'] = 'https://openalex.org/F4320338075'
last_dict['State of Hungary'] = 'https://openalex.org/F4320328860'
last_dict['Alfred P. Sloan Research Fellowship'] = 'https://openalex.org/F4320306151'
last_dict['Russian Fund for Basic Research'] = 'https://openalex.org/F4320321079'
last_dict['Cotton RandD Corp'] = 'https://openalex.org/F4320320417'
last_dict['Merck-Serono'] = 'https://openalex.org/F4320307776'
last_dict['JST CREST, Japan'] = 'https://openalex.org/F4320338075'
last_dict['Brain Korea 21 Project'] = 'https://openalex.org/F4320336052'
last_dict['PAPIIT-UNAM'] = 'https://openalex.org/F4320335100'
last_dict['Taishan Scholar Program of Shandong Province'] = 'https://openalex.org/F4320326189'
last_dict['US Public Health Service'] = 'https://openalex.org/F4320332505'
last_dict['Piramal Imaging'] = 'https://openalex.org/F4320307771'
last_dict['Stanford Graduate Fellowship'] = 'https://openalex.org/F4320308507'
last_dict['Astra-Zeneca'] = 'https://openalex.org/F4320307770'
last_dict['FAPERJ (Brazil)'] = 'https://openalex.org/F4320322749'
last_dict['CAPES, Brazil'] = 'https://openalex.org/F4320321091'
last_dict['European Communities'] = 'https://openalex.org/F4320338352'
last_dict['National Scientific Foundation of China'] = 'https://openalex.org/F4320321001'
last_dict['Sloan Research Fellowship'] = 'https://openalex.org/F4320306151'
last_dict['Fondo de Investigaciones Sanitarias'] = 'https://openalex.org/F4320334923'
last_dict['DIVISION OF HEART AND VASCULAR DISEASES'] = 'https://openalex.org/F4320337338'
last_dict['Division Of Behavioral and Cognitive Sci'] = 'https://openalex.org/F4320337405'
last_dict['COST action'] = 'https://openalex.org/F4320320366'
last_dict['National Science and Technology Pillar Program'] = 'https://openalex.org/F4320335774'
last_dict['National Key Technology Support Program'] = 'https://openalex.org/F4320335774'
last_dict['China Post-doctoral Science Foundation'] = 'https://openalex.org/F4320321543'
last_dict['Alzheimers Research UK'] = 'https://openalex.org/F4320321081'
last_dict['BLRD VA'] = 'https://openalex.org/F4320309842'
last_dict['Texas AM University'] = 'https://openalex.org/F4320310164'
last_dict['NSF CAREER award'] = 'https://openalex.org/F4320306076'
last_dict['DIVISION OF CANCER PREVENTION AND CONTROL'] = 'https://openalex.org/F4320309684'
last_dict['Fondo de Investigacion Sanitaria'] = 'https://openalex.org/F4320334923'
last_dict['MICINN (Spain)'] = 'https://openalex.org/F4320322930'
last_dict['Swiss NSF'] = 'https://openalex.org/F4320320924'
last_dict['Research Grant Council of Hong Kong'] = 'https://openalex.org/F4320321592'
last_dict['state budget of the Czech Republic'] = 'https://openalex.org/F4320321006'
last_dict['Solvay SA'] = 'https://openalex.org/F4320333783'

last_dict['PAPIIT'] = 'https://openalex.org/F4320335100'
last_dict['Beijing Municipal Science and Technology Project'] = 'https://openalex.org/F4320335843'
last_dict['Region Centre-Val de Loire'] = 'https://openalex.org/F4320325797'
last_dict['Programme of Introducing Talents of Discipline to Universities'] = 'https://openalex.org/F4320335785'
last_dict['National Science and Technology Support Program of China'] = 'https://openalex.org/F4320335961'
last_dict['Shenzhen Science and Technology Innovation Committee'] = 'https://openalex.org/F4320326705'
last_dict['ARC Future Fellowship'] = 'https://openalex.org/F4320334704'
last_dict['Guangdong Natural Science Funds for Distinguished Young Scholar'] = 'https://openalex.org/F4320327517'
last_dict['FWO-Vlaanderen'] = 'https://openalex.org/F4320321730'
last_dict['Hong Kong RGC'] = 'https://openalex.org/F4320321592'
last_dict['National Institute of Drug Abuse'] = 'https://openalex.org/F4320337347'
last_dict['Programa de Apoyo a Proyectos de Investigacion e Innovacion Tecnologica (PAPIIT)'] = 'https://openalex.org/F4320335100'


for k, v in last_dict.items():
    v = str(v)
    last_dict[k] = v.split('F')[-1]

last_dict = {k.strip():str(v).split('F')[-1] for k, v in last_dict.items()}

In [19]:
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(last_dict))

u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'manual'

print(u_agencies.shape)
print(u_agencies.id.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(7420729, 10)
0.9852023972307842


0.6298905730985602

In [20]:
u_agencies['agency'] = u_agencies['display_name'].fillna(u_agencies['all_names_lower'])

### Prepare data for detecting substrings

In [21]:
found_agency = institutions.all_names.explode().dropna().unique()

found_agency = sorted(found_agency, key=len, reverse=True)

found_agency = [x for x in found_agency if  " " in x]    

found_agency_lower = [x.lower() for x in found_agency]

found_agency_lower_asc = sorted(found_agency_lower, key=len)

found_name = u_agencies['grant_agency']

found_name = sorted(found_name, key=len, reverse=True)

found_name = [x for x in found_name if  " " in x]

found_name_lower = [x.lower() for x in found_name]

found_name_lower_asc = sorted(found_name_lower, key=len)



In [22]:
def longest_string(s1, s2):
    if isinstance(s1, str) and isinstance(s2, str):
        return max(s1, s2, key=len)
    elif isinstance(s1, str):
        return s1
    elif isinstance(s2, str):
        return s2
    else:
        return np.nan

def build_automaton(words):
    A = ahocorasick.Automaton()
    for word in words:
        A.add_word(word, word)
    A.make_automaton()
    return A


def find_matches_end(text, automatons):
    matches = []
    for A in automatons:
        for end_index, word in A.iter(text):
            start_index = end_index - len(word) + 1
            # Check for word boundaries
            if end_index == len(text) - 1 and (start_index == 0 or not text[start_index - 1].isalnum()):
                matches.append(word)
        if matches:
            return max(matches, key=len, default=None)
    return None

def find_longest_match(text, automatons):
    matches = []
    for A in automatons:
        for end_index, word in A.iter(text):
            start_index = end_index - len(word) + 1
            # Check for word boundaries
            if (start_index == 0 or not text[start_index - 1].isalnum()) and \
            (end_index == len(text) - 1 or not text[end_index + 1].isalnum()):
                matches.append(word)
        if matches:
            return max(matches, key=len, default=None)
    return None


def find_matches_start(text, automatons):
    matches = []
    for A in automatons:
        for end_index, word in A.iter(text):
            if end_index - len(word) + 1 == 0 and (end_index + 1 == len(text) or not text[end_index + 1].isalnum()):
                matches.append(word)
        if matches:
            return max(matches, key=len, default=None)
    return None

def shortest_matching_substring(substrings, target_string):
    # Filter the substrings that are present in the target_string
    for s in substrings:
        if len(s) < len(target_string):
            continue
        if target_string in s:
            return s
    return None

## Detecting substrings

Here we detect the longest substrings in the start, end and anywhere in the names

In [23]:
A = build_automaton(found_agency_lower)

u_agencies['agency_in_name_rev'] = u_agencies['grant_agency_lower'].progress_apply(lambda x: find_matches_end(x, [A]))
u_agencies['agency_in_name_start'] = u_agencies['grant_agency_lower'].progress_apply(lambda x: find_matches_start(x, [A]))
u_agencies['agency_in_name'] = u_agencies['grant_agency_lower'].progress_apply(lambda x: find_longest_match(x, [A]))

def split_sentences(sentences_list, num_splits=10):
    split_size = len(sentences_list) // num_splits
    return [sentences_list[i * split_size: (i + 1) * split_size] for i in range(num_splits)]

# sentence_splits = split_sentences(found_name_lower)
# automatons = [build_automaton(split) for split in sentence_splits]

# test = (institutions.explode('all_names_lower')
#          .sort_values('all_names_lower', key=lambda x: x.str.len(), ascending=False)
# ).assign(name_in_agency=lambda x: x['all_names_lower'].progress_apply(lambda x: find_longest_match(x, automatons)),
#          name_in_agency_rev=lambda x: x['all_names_lower'].progress_apply(lambda x: find_matches_end(x, automatons)),
#           name_in_agency_start=lambda x: x['all_names_lower'].progress_apply(lambda x: find_matches_start(x, automatons)))

# u_agencies = u_agencies.merge(test[['all_names_lower', 'name_in_agency', 'name_in_agency_rev', 'name_in_agency_start']],
#                               left_on='grant_agency_lower', right_on='all_names_lower',
#                                 how='left', suffixes=('', '_y')).drop(columns='all_names_lower_y')


100%|██████████| 7420729/7420729 [00:19<00:00, 376816.66it/s]


In [24]:
#rename agency in name
u_agencies.rename(columns={'agency_in_name': 'agency_in_name_search'}, inplace=True)
#u_agencies.rename(columns={'name_in_agency': 'name_in_agency_search'}, inplace=True)

In [25]:
u_agencies['agency_in_name'] = u_agencies[['agency_in_name_start', 'agency_in_name_rev']].apply(
    lambda x: longest_string(x['agency_in_name_start'], x['agency_in_name_rev']), axis=1)
# u_agencies['name_in_agency'] = u_agencies[['name_in_agency_start', 'name_in_agency_rev']].apply(
#     lambda x: longest_string(x['name_in_agency_start'], x['name_in_agency_rev']), axis=1)

### Substrings in the start or end of names

In [26]:
test = u_agencies[ u_agencies.id.isna() &
    ((u_agencies.agency_in_name.str.len() / u_agencies.grant_agency.str.len()) > 0.5)][['grant_agency', 'agency_in_name']]

In [27]:
u_agencies['agency_in_name'][((u_agencies.agency_in_name.str.len() / 
             u_agencies.grant_agency.str.len()) < 0.5)] = np.nan

/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_2275/1057552185.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  u_agencies['agency_in_name'][((u_agencies.agency_in_name.str.len() /


In [28]:
name_to_id = (u_agencies[['agency_in_name']].dropna()
              .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
                      left_on='agency_in_name', right_on='all_names_lower', how='inner')
                      .drop_duplicates()
)


u_agencies = u_agencies.merge(name_to_id[['agency_in_name', 'id']],
    left_on='agency_in_name', right_on='agency_in_name', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'start_or_end'

u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['agency_in_name'])


In [29]:
# name_to_id = (u_agencies[['agency_in_name']].dropna()
#               .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
#                       left_on='agency_in_name', right_on='all_names_lower', how='inner')
#                       .drop_duplicates(subset='agency_in_name'))

# #map agency in name to id using dict
# name_to_id = name_to_id.set_index('agency_in_name')['id'].dropna().to_dict()

# #fill missing ids with mapped ids
# u_agencies['id'] = u_agencies['id'].fillna(u_agencies['agency_in_name'].map(name_to_id))
# u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'start_or_end'

# u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['agency_in_name'])



### Substrings anywhere in the names

In [30]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(7852529, 15)
0.7979811344854633 0.798457286818043


0.761151616452448

In [31]:
u_agencies['agency_in_name_search'][ u_agencies.id.isna() & ((u_agencies.agency_in_name_search.str.len()
                     / u_agencies.grant_agency.str.len()) < 0.5)] = np.nan

In [32]:
name_to_id = (u_agencies[['agency_in_name_search']].dropna()
              .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
                      left_on='agency_in_name_search', right_on='all_names_lower', how='inner')
                      .drop_duplicates()
)


u_agencies = u_agencies.merge(name_to_id[['agency_in_name_search', 'id']],
    left_on='agency_in_name_search', right_on='agency_in_name_search', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'substring'

u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['agency_in_name_search'])


In [30]:
# name_to_id = (u_agencies[['agency_in_name_search']].dropna()
#               .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
#                     left_on='agency_in_name_search', right_on='all_names_lower', how='inner')
#                     .drop_duplicates(subset='agency_in_name_search'))

# #map agency in name to id using dict
# name_to_id = name_to_id.set_index('agency_in_name_search')['id'].dropna().to_dict()

# u_agencies['id'] = u_agencies['id'].fillna(u_agencies['agency_in_name_search'].map(name_to_id))
# u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'substring'

# u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['agency_in_name_search'])



In [33]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(7974664, 15)
0.7568889422802014 0.7573362338526112


0.7744631670171022

### Matching acronyms

In [34]:
acronyms = (institutions.explode([ 'display_name_acronyms'])
                              [['display_name_acronyms', 'id']].dropna().drop_duplicates())

u_agencies = u_agencies.merge(acronyms,
    left_on='abbrev', right_on='display_name_acronyms', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'acronyms'

u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)

In [35]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(9272088, 16)
0.5954032144647462 0.7912858462948151


0.8054762067157771

### Document cluster ids
Based on the document clusters if we found and id we assign all documents in the cluster the same id

In [36]:
#fill all ids with the corresponding id based associated component
component_id_map = u_agencies.groupby('component')['id'].first()
component_name_map = u_agencies.groupby('component')['agency'].first()


u_agencies['id'] = u_agencies['id'].fillna(u_agencies['component'].map(component_id_map))
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'
u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['component'].map(component_name_map))



In [37]:
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

0.5865705761205028 0.7793736427005439


0.8165594249213131

In [36]:
# u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(last_dict))
# u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'manual'

# component_id_map = u_agencies.groupby('component')['id'].first()


# u_agencies['id'] = u_agencies['id'].fillna(u_agencies['component'].map(component_id_map))
# u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'


In [38]:
u_agencies_missing = u_agencies[u_agencies['id'].isna()]
u_agencies_missing.shape

(5438734, 16)

In [39]:
x = u_agencies_missing['counts'].cumsum()

test = u_agencies_missing[((u_agencies['counts'].sum()*1) - u_agencies[~u_agencies.id.isna()]['counts'].sum()) > x]
test = test[test.counts >= 1000]
test

,grant_agency,counts,grant_agency_lower,abbrev,component,display_name_lower,display_name,match_type,all_names_lower,agency,agency_in_name_rev,agency_in_name_start,agency_in_name_search,agency_in_name,display_name_acronyms,id
3692,Egyptian Knowledge Bank (EKB),1708,egyptian knowledge bank (ekb),EKB,1719,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
3963,Araclon Biotech,1594,araclon biotech,NaN,1861,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
4654,"CereSpir, Inc.",1297,"cerespir, inc.",NaN,2253,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
4820,Cogstate,1227,cogstate,NaN,2363,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
5534,"MSSR, Slovakia",1033,"mssr, slovakia",NaN,2780,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
5643,Municipality of Rotterdam,1007,municipality of rotterdam,NaN,2849,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None


In [40]:
res_dict = {}
for abbrev in test.abbrev.dropna().unique():
    res = query_openalex_funders(abbrev)[0]
    try:
        res_dict[abbrev] = res['id']
    except:
        pass

for k, v in res_dict.items():
    v = str(v)
    res_dict[k] = v.split('F')[-1]

In [41]:
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['abbrev'].map(res_dict))
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'manual'

component_id_map = u_agencies.groupby('component')['id'].first()
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['component'].map(component_id_map))
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'


# Deduplication using LSH

In [42]:
u_agencies_missing = u_agencies[u_agencies['id'].isna()]
u_agencies_missing.shape

(5438734, 16)

In [46]:
for r in query_openalex_funders("Chinese Academy of Sciences"):
    print(r['display_name'], r['alternate_titles'])

Chinese Academy of Sciences ['CAS', '中国科学院']
Youth Innovation Promotion Association of the Chinese Academy of Sciences ['CAS YIPA', 'YIPA CAS', 'Youth Innovation Promotion Association, Chinese Academy of Sciences', 'Youth Innovation Promotion Association of Chinese Academy of Sciences', '中国科学院青年创新促进会', 'Youth Innovation Promotion Association, CAS', 'Youth Innovation Promotion Association CAS']
University of Chinese Academy of Sciences ['中国科学院大学', 'University of the Chinese Academy of Sciences', 'UCAS', 'Graduate University of Chinese Academy of Sciences']
West Light Foundation of the Chinese Academy of Sciences ['West Light Foundation of Chinese Academy of Sciences', 'West Light Foundation of CAS']
Chinese Academy of Sciences Key Project ['CAS Key Project']
Hefei Science Center, Chinese Academy of Sciences ['Hefei Science Center of CAS', 'Scientific Research Grant of Hefei Science Center of CAS', 'CAS Hefei Science Center', 'Hefei Science Center, CAS']
Haixi Institute, Chinese Academy 

### We probably want to find matches for agencies with more than 100 papers using chatGPT

For now these are left unmatched

In [43]:
x = u_agencies_missing['counts'].cumsum()

test = u_agencies_missing[((u_agencies['counts'].sum()*1) - u_agencies[~u_agencies.id.isna()]['counts'].sum()) > x]
test = test[test.counts >= 1000]
test

,grant_agency,counts,grant_agency_lower,abbrev,component,display_name_lower,display_name,match_type,all_names_lower,agency,agency_in_name_rev,agency_in_name_start,agency_in_name_search,agency_in_name,display_name_acronyms,id
3692,Egyptian Knowledge Bank (EKB),1708,egyptian knowledge bank (ekb),EKB,1719,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
3963,Araclon Biotech,1594,araclon biotech,NaN,1861,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
4654,"CereSpir, Inc.",1297,"cerespir, inc.",NaN,2253,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
4820,Cogstate,1227,cogstate,NaN,2363,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
5534,"MSSR, Slovakia",1033,"mssr, slovakia",NaN,2780,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None
5643,Municipality of Rotterdam,1007,municipality of rotterdam,NaN,2849,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,None


In [44]:
#test[['grant_agency']].to_csv("plus_1000_for_gpt.csv", index=False)

In [45]:
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

0.5865705761205028 0.7793736427005439


0.8165594249213131

In [46]:
u_agencies_exact = u_agencies.dropna(subset=['id','agency'])
print(u_agencies_exact.shape)
u_agencies_exact = u_agencies_exact.drop_duplicates(subset=['id', 'agency'])
print(u_agencies_exact.shape)

(2045618, 16)
(163932, 16)


## this block takes about about 1-2 hours to run

In [46]:

minhash_file = 'institutions_cand_minhashes.pkl'

queries = u_agencies_exact.agency.str.lower()
cands = u_agencies_missing.grant_agency.str.lower()

# Serialize MinHashes and store them to a file
with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(cands.index, cands), total=len(cands), desc="Adding MinHashes to LSH"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)


minhash_file = 'institutions_query_minhashes.pkl'

with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(queries.index, queries), total=len(queries), desc="Querying Candidate Strings"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)



Querying Candidate Strings: 100%|██████████| 163932/163932 [01:46<00:00, 1538.15it/s]


In [47]:

# Create the LSH object with threshold for near-duplicates
lsh = MinHashLSH(threshold=0.9, num_perm=128)

minhash_file = 'institutions_cand_minhashes.pkl'

# Load MinHashes from file and insert them into the LSH
ref_minhashes = {}
i = 0
with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            lsh.insert(f"ref_{idx}", lean_minhash)
            i += 1
    except EOFError:
        pass 

minhash_file = 'institutions_query_minhashes.pkl'

candidate_results = []
with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            result = lsh.query(lean_minhash)
            candidate_results.append({'cand_id': idx, 'duplicates': result})
    except EOFError:
        pass

In [48]:
results_df = pd.DataFrame(candidate_results)
results_df['duplicates'] = results_df['duplicates'].apply(lambda x: [int(y.split('_')[1]) for y in x])

results_df = results_df[results_df['duplicates'].apply(len) > 0]


In [49]:
results_df['doc_text'] = results_df['cand_id'].apply(lambda x: u_agencies_exact['agency'][int(x)])
results_df['duplicates_text'] = results_df['duplicates'].apply(lambda x: [u_agencies_missing['grant_agency'][int(i)] for i in x])

In [50]:
results_df = results_df.explode(['duplicates_text', 'duplicates']).dropna(subset=['duplicates_text'])
results_df = results_df.merge(u_agencies[['id', 'agency']], left_on='cand_id', right_index=True, how='left')

In [51]:
# results_df.drop_duplicates(subset=['duplicates_text'], inplace=True)

# name_to_id = results_df.set_index('duplicates_text')['id'].to_dict()
# name_to_agency = results_df.set_index('duplicates_text')['agency'].to_dict()

# u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(name_to_id))

# u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'deduplication'
# u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['grant_agency'].map(name_to_agency))

In [52]:
u_agencies = u_agencies.merge(results_df,
    left_on='grant_agency', right_on='duplicates_text', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies['agency'] = u_agencies['agency_x'].fillna(u_agencies['agency_y'])

u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'deduplication'

u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.drop(columns=['agency_x', 'agency_y'], inplace=True)

In [53]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(9313003, 20)
0.580405697281532 0.7723617183415489


0.819775910262217

In [54]:
component_id_map = u_agencies.groupby('component')['id'].first()
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['component'].map(component_id_map))
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'
component_name_map = u_agencies.groupby('component')['agency'].first()
u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['component'].map(component_name_map))

In [55]:
test = u_agencies.drop_duplicates(subset=['grant_agency_lower'])
print(test.id.isna().mean(), test.agency.isna().mean())
(test[~test.id.isna()]['counts'].sum()) / test['counts'].sum()

0.7301789707539553 0.8002869543616603


0.7349068945897578

In [56]:
print(test[(test.counts >= 1000)].id.isna().mean())
print(test[(test.counts >= 100) & (test.counts <=1000)].id.isna().mean())
print(test[(test.counts >= 10) & (test.counts <=100)].id.isna().mean())
print(test[(test.counts >= 1) & (test.counts <=10)].id.isna().mean())

0.002068252326783868
0.21852870115804382
0.5823765394649032
0.7372895342916861


In [ ]:
print(test[(test.counts >= 1000)].id.isna().sum())
print(test[(test.counts >= 100) & (test.counts <=1000)].id.isna().sum())
print(test[(test.counts >= 10) & (test.counts <=100)].id.isna().sum())
print(test[(test.counts >= 1) & (test.counts <=10)].id.isna().sum())

6
5246
153589
5262320


In [70]:
test_ = test.query("counts >=500 & counts <= 1000 & id.isna()")

In [101]:
print(u_agencies.drop_duplicates('grant_agency').match_type.value_counts())

match_type
nan               5402657
start_or_end      1044750
acronyms           515304
substring          230235
doc_clustering      84560
display_exact       44870
all_exact           39376
deduplication       33414
manual               3920
Name: count, dtype: int64


In [78]:
# id column to string to int
u_agencies['id'] = u_agencies['id'].astype('float64').astype('Int64')

In [86]:
final_map = (u_agencies.merge(
        institutions[['id', 'display_name', 'ids:ror', 'source']], left_on='id', right_on='id', how='left')
[['grant_agency', 'id', 'counts', 'ids:ror', 'source', 'match_type', 'display_name_y']]).drop_duplicates()

In [95]:
final_map = final_map.assign(match_type = final_map.match_type.replace('nan', None))

In [99]:
final_map.drop_duplicates('grant_agency').query("(~match_type.isna()) & display_name_y.isna()")

,grant_agency,id,counts,ids:ror,source,match_type,display_name_y
819,Canadian Foundation for Innovation,4320320995,6761,NaN,NaN,manual,NaN
1423,European Union's Horizon 2020 research and inn...,4321443650,3616,NaN,NaN,manual,NaN
6031,Marie Curie Actions,4321443650,724,NaN,NaN,manual,NaN
8958,European Union's Horizon research and innovati...,4321443650,461,NaN,NaN,doc_clustering,NaN
9392,European Union's Horizon 2020 research and inn...,4321443650,439,NaN,NaN,doc_clustering,NaN
...,...,...,...,...,...,...,...
9200741,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN
9225433,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN
9239285,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN
9246203,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN


In [100]:
print((~final_map.drop_duplicates('grant_agency').isna()).sum())

grant_agency      7399086
id                1996429
counts            7399086
ids:ror           1780822
source            1996132
match_type        1996429
display_name_y    1996132
dtype: int64


In [76]:

# Load the data file (replace 'filename.csv' with the path to your file)
filename = 'v1.55-2024-10-31-ror-data_schema_v2.csv'

# Read the CSV file into a pandas DataFrame
ror = pd.read_csv(
    filename,
    dtype=str,  # Treat all columns as strings to handle mixed data types
    parse_dates=[
        'admin.created.date', 
        'admin.last_modified.date'
    ],  # Automatically parse date columns
    keep_default_na=False  # Prevent empty strings from being converted to NaN
)


In [77]:
in_dimensions = pl.read_parquet('funders_dimensions.parquet').with_columns(pl.lit('True').alias('in_dimensions')).to_pandas()
ror = ror.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')

final_map = final_map.merge(ror[['id', 'types', 'relationships','funder', 'in_dimensions']], left_on="ids:ror", right_on='id', how='left', suffixes=("","_ror"))

In [79]:
final_map[final_map.id.isna()].to_feather('missing_agencies.feather')

In [80]:
final_map[~(final_map.id.isna())].to_feather('institutions_processed_duplicates.feather')